# 11 · Look-up tables without interpolation

Notebook 10's tables do little arithmetic, but they are slow as a graph: reading a 4-D table
between its points is simplex interpolation - sort the four fractions, find five corners,
weigh them - and as ONNX operators that is some sixty full-size tensors written to memory and
read back, per stage. This notebook tries two ways to read **one entry, at an exact integer
address**, with no interpolation, no sort and no weights to work out.

**A binned table** (`S`, `SDY x2`) - the entry is the answer:

    LR luma ──► 4 taps per kernel and rotation ──► centre level + 3 differences, each binned by a small index table
                                                   ──► their sum is the row ──► one read ──► turn back, average ──► HR luma correction

- **The address is the centre and the differences from it.** A table that is never
  interpolated needs a row for every input it can see, and four 8-bit taps are four billion.
  But neighbouring pixels are mostly alike - notebook 10 found 85% of its reads within one
  table step of the diagonal - so the row is the centre's brightness in 8 levels, and each of
  the three other taps' **difference** from it in 19 bins: fine near 0 (0, ±1, ±2, ±3-4,
  ±5-7, ...) and coarse far from it (±61-255). That is 8 x 19³ = 54,872 rows a kernel.
- **The bins are a table too.** A tap's rounded value (or difference) is an index into a small
  table of integers that holds its bin already multiplied by its place in the row, so four
  pixels become a row with one convolution, a cast, one read of the small table and a sum.
- **The rest is notebook 10's**: MuLUT's kernels S, D and Y, each in four rotations sharing one
  table; a second stage on the first one's output.

**A 3x3 pattern filter** (`3x3 pattern`) - the entry is a filter:

    LR luma ──► one 3x3 convolution: the 8 differences from the centre ──► their signs: the pattern ──► one read: its filter
                                                                        └──────────────────────────────────► × ──► HR luma correction

- **Every combination has a row.** The pattern is each of the 8 neighbours darker than the
  centre, the same (within 2 of 255 steps) or brighter: 3⁸ = 6,561 of them, all in the table.
- **The row is a filter**: for each of the four x2 sub-pixels, a weight per neighbour and a
  bias, applied to the actual differences - 36 multiply-adds. Since the filter is linear in
  the pixels, a strong edge gets a proportionally strong correction.

Both are trained as tables: the entries are the parameters - there is no network to fill them
from - and they start at zero, so untrained either model is bicubic. Both correct luma only.

Steps:

1. **Data** - the patch set, and what each validation patch shows.
2. **Model** - both designs, and the checks that their addresses are exact.
3. **Speed** - counted from the exported graphs, against notebook 7's model and notebook
   10's interpolated tables.
4. **Train** - `S` at three seeds, which is the ruler, the others once; results overall, by
   content, and pictures.
5. **Save** - the best, and check it loads back the same.
6. **Export** - every model, at the client's tile, in float32.

Needs: `00_data_preprocess.ipynb`. Writes `checkpoints/upscale_fastlut.pt` (the best) and
each run's training to `checkpoints/fastlut_runs/`.

In [ ]:
"""Settings."""

import copy

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset

from helpers import metrics, models, speed, tiles, training, webexport
from helpers.training import DATA_DIR, DEVICE

CHECKPOINT = training.CHECKPOINT_DIR / "upscale_fastlut.pt"   # the best
RUN_DIR = training.CHECKPOINT_DIR / "fastlut_runs"            # each run's training, saved every epoch
SPEED_DIR = training.CHECKPOINT_DIR / "speed"                 # graphs exported only to be counted
BEFORE = training.CHECKPOINT_DIR / "upscale_strategies.pt"    # notebook 4's luma model: the one to beat

CENTRE_LEVELS = 8                         # the centre's brightness, in 8 steps of 32
EDGES = (1, 2, 3, 5, 8, 13, 21, 35, 61)   # where a difference's bins start, in 8-bit steps: 0, 1, 2, 3-4, 5-7, ...
DEAD_ZONE = 2.5     # 8-bit steps: a neighbour within 2 of the centre is "the same" in a 3x3 pattern
MAX_EPOCHS = 500     # a safety limit; each training stops once it has converged
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
SEEDS = (0, 1, 2)    # S trained three times: how far luck alone moves it

STEP_WIDTH, STEP_HEIGHT = 320, 180       # LR pixels one client tile adds to the picture (notebook 5)
FRAME_WIDTH, FRAME_HEIGHT = 1920, 1080   # the whole-frame graph
ESTIMATE_ON = {"Iris Xe": "Intel Iris Xe, 96 EU (laptop iGPU)", "GTX 1650": "NVIDIA GTX 1650",
               "RTX 3060": "NVIDIA RTX 3060"}

training.seed_everything(SEEDS[0])

## 1. Data

The patch set of `00_data_preprocess.ipynb` - photos, videos and drawn desktop screens through
the stream's degradation - as notebooks 8 to 10 see it, with results split by what a patch
shows.

In [ ]:
manifest = training.load_manifest()
training.describe(manifest)
SCALE = manifest["scale"]
train_loader, val_loader = training.loaders(DATA_DIR, manifest, BATCH_SIZE, val_batch_size=128)
trim = training.trimmer(manifest)


def content(source):
    """What a source shows: a drawn desktop screen, a photo (as a moving clip) or a video."""
    if source["dataset"] == "screens":
        return "screen"
    return "photo" if "motion" in source else "video"


by_content = {}   # what a patch shows -> the indices of its validation patches
for source in manifest["splits"]["val"]["sources"]:
    first, end = source["patches"]
    by_content.setdefault(content(source), []).extend(range(first, end))
print("val by content: " + ", ".join(f"{name} {len(indices):,}" for name, indices in by_content.items()))

## 2. Model

**The binned table.** Per stage, every kernel in each of four rotations - a *group* - reads four taps: the centre
and three around it.

- **Read.** One frozen convolution writes, per group, the centre and the three differences
  from it in 8-bit steps, plus 0.5 (and 255.5 for a difference) so that dropping the fraction
  rounds and nothing is negative. A cast to integers drops the fraction.
- **Bin.** Each channel adds where its part of the index table starts, and reads it: a
  centre value gives its level times 19³ (plus where its kernel's table starts), a
  difference its bin times 19², 19 or 1.
- **Sum.** The four numbers add up to the row - three integer `Add`s, not a `ReduceSum`, which
  the browser's WebGPU provider has no integer kernel for. Every row is a different combination of
  bins, and every combination has a row.
- **Answer.** One read of the table gives the group's answer: one number in an earlier
  stage (a correction of the LR luma the next stage reads), the four x2 sub-pixels in the
  last. A frozen 1x1 convolution turns each rotation back, averages the rotations and adds
  the kernels, as in notebook 10.

**The 3x3 pattern filter.** One frozen 3x3 convolution writes the 8 differences from the
centre. Scaled so that the dead zone is half a step, rounded and clipped to -1..1, they are
the pattern's 8 base-3 digits; a 1x1 convolution adds them up in two halves of four (each
0..80), and the halves meet in float32, where every row number is exact, before the cast:
  the browser's WebGPU provider has no integer `Mul`. Rounding is `floor(x + 0.5)`, which it
  runs, where `Round` it does not; on 8-bit pixels the two never differ, since no scaled
  difference lands on a half. The row read is the filter,
applied to the differences.

In both, luma is clamped to 0..1 first, so whatever a graph is fed, every address stays
inside its table.

In [ ]:
LUMA = torch.tensor(models.LUMA_709)
ROTATIONS = 4
KERNELS = {   # a kernel's four taps, (down, across) from the pixel it answers for: MuLUT's
    "S": ((0, 0), (0, 1), (1, 0), (1, 1)),   # SR-LUT's 2x2 square
    "D": ((0, 0), (0, 2), (2, 0), (2, 2)),   # the square, dilated
    "Y": ((0, 0), (1, 1), (1, 2), (2, 1)),   # the diagonal neighbour and two beside it
}
BINS = 2 * len(EDGES) + 1   # 19: the bins either side of 0, and 0


def rotate(offset, turns):
    """An offset (down, across) turned by `turns` quarter turns."""
    dy, dx = offset
    for _ in range(turns):
        dy, dx = dx, -dy
    return dy, dx


def frozen(conv, weight, bias=None):
    """`conv` with these weights (and bias), never trained."""
    with torch.no_grad():
        conv.weight.copy_(weight.reshape(conv.weight.shape))
        if bias is not None:
            conv.bias.copy_(bias)
    for parameter in conv.parameters():
        parameter.requires_grad_(False)
    return conv


def difference_bin(d):
    """The bin of an 8-bit difference -255..255: 0 for the most negative, BINS // 2 for 0."""
    magnitude = torch.bucketize(d.abs(), torch.tensor(EDGES), right=True)
    return torch.where(d >= 0, magnitude, -magnitude) + len(EDGES)


class FastLut(models.Upscaler):
    """Bicubic colour, and a luma correction read from tables at one exact address each."""

    def __init__(self, kernels="S", stages=1, scale=2):
        super().__init__(scale, skip=5)
        self.kernels, self.stages = kernels, stages
        G = self.groups = len(kernels) * ROTATIONS
        self.outs = [1] * (stages - 1) + [scale * scale]   # what a group answers, per stage
        reach = max(abs(v) for name in kernels for tap in KERNELS[name] for v in tap)
        self.halo = max(stages * reach, self.skip_size // 2)
        self.rows = CENTRE_LEVELS * BINS ** 3   # per kernel
        self.to_luma = frozen(nn.Conv2d(3, 1, 1, bias=False), LUMA)

        # the index table: each kernel's 256 centre values -> level * BINS³ + where its table
        # starts; then, for taps 1-3, the 511 differences -> bin * BINS^(3 - tap)
        level = torch.arange(256) * CENTRE_LEVELS // 256
        centre = torch.cat([level * BINS ** 3 + k * self.rows for k in range(len(kernels))])
        d = torch.arange(-255, 256)
        taps = torch.cat([difference_bin(d) * BINS ** (3 - t) for t in (1, 2, 3)])
        self.register_buffer("index", torch.cat([centre, taps]).to(torch.int32), persistent=False)
        offset = torch.zeros(G * 4, dtype=torch.int32)   # where each channel's part of the index table starts
        for g in range(G):
            offset[g * 4] = g // ROTATIONS * 256
            for t in (1, 2, 3):
                offset[g * 4 + t] = len(centre) + (t - 1) * 511
        self.register_buffer("offset", offset.view(1, G * 4, 1, 1), persistent=False)

        size, middle = 2 * reach + 1, (scale - 1) / 2
        self.reads, self.combines = nn.ModuleList(), nn.ModuleList()
        for out in self.outs:
            # read: channel g * 4 + t is the centre (t = 0) or tap t minus the centre, in 8-bit steps
            read = torch.zeros(G * 4, 1, size, size)
            for k, name in enumerate(kernels):
                for r in range(ROTATIONS):
                    g = k * ROTATIONS + r
                    (y0, x0), *others = [rotate(tap, r) for tap in KERNELS[name]]
                    read[g * 4, 0, reach + y0, reach + x0] = 255
                    for t, (y, x) in enumerate(others, start=1):
                        read[g * 4 + t, 0, reach + y, reach + x] += 255
                        read[g * 4 + t, 0, reach + y0, reach + x0] -= 255
            bias = torch.tensor([0.5, 255.5, 255.5, 255.5]).repeat(G)   # dropping the fraction rounds
            self.reads.append(frozen(nn.Conv2d(1, G * 4, size, padding=reach), read, bias))
            # combine: a group's answer o is the sub-pixel o turned by its rotation; rotations
            # are averaged and kernels added
            combine = torch.zeros(out, G * out)
            for g in range(G):
                for o in range(out):
                    dy, dx = rotate((o // scale - middle, o % scale - middle), g % ROTATIONS)
                    to = round(dy + middle) * scale + round(dx + middle) if out > 1 else 0
                    combine[to, g * out + o] = 1 / ROTATIONS
            self.combines.append(frozen(nn.Conv2d(G * out, out, 1, bias=False), combine))
        # the tables, all kernels of a stage end to end; untrained, every answer is 0: bicubic
        self.tables = nn.ParameterList(nn.Parameter(torch.zeros(len(kernels) * self.rows, out)) for out in self.outs)

    def address(self, stage, luma):
        """(N, 1, H, W) luma -> (N, groups, H, W) table rows."""
        n, _, h, w = luma.shape
        steps = self.reads[stage](luma).to(torch.int32) + self.offset
        parts = self.index.index_select(0, steps.flatten()).view(n, self.groups, 4, h, w)
        return parts[:, :, 0] + parts[:, :, 1] + parts[:, :, 2] + parts[:, :, 3]   # Adds: WebGPU has no integer ReduceSum

    def residual(self, x):
        luma = self.to_luma(x).clamp(0, 1)   # whatever comes in, every address stays inside its table
        for stage, table in enumerate(self.tables):
            row = self.address(stage, luma)
            n, G, h, w = row.shape
            answers = table.index_select(0, row.flatten()).view(n, G, h, w, -1).permute(0, 1, 4, 2, 3)
            correction = self.combines[stage](answers.reshape(n, -1, h, w))
            if stage < self.stages - 1:
                luma = (luma + correction).clamp(0, 1)
        return torch.cat([correction, correction, correction], dim=1)   # on every colour: Y moves


NEIGHBOURS = [(dy, dx) for dy in (-1, 0, 1) for dx in (-1, 0, 1) if (dy, dx) != (0, 0)]


class PatternFilter(models.Upscaler):
    """Bicubic colour, and a luma correction by a filter looked up for the 3x3 neighbourhood's pattern."""

    def __init__(self, dead_zone=DEAD_ZONE, scale=2):
        super().__init__(scale, skip=5)
        self.dead_zone = dead_zone
        self.halo = max(1, self.skip_size // 2)
        self.to_luma = frozen(nn.Conv2d(3, 1, 1, bias=False), LUMA)
        # the one 3x3 convolution: neighbour minus centre, for each of the 8 neighbours
        weight = torch.zeros(8, 1, 3, 3)
        for i, (dy, dx) in enumerate(NEIGHBOURS):
            weight[i, 0, 1 + dy, 1 + dx] = 1
            weight[i, 0, 1, 1] = -1
        self.differences = frozen(nn.Conv2d(1, 8, 3, padding=1, bias=False), weight)
        # the 8 signs (-1, 0, 1) as base-3 digits, added up in two halves of four: each 0 .. 80
        halves = torch.zeros(2, 8)
        halves[0, :4] = halves[1, 4:] = torch.tensor([1.0, 3, 9, 27])
        self.to_halves = frozen(nn.Conv2d(8, 2, 1), halves, torch.tensor([40.0, 40.0]))
        # per pattern and sub-pixel: 8 weights and a bias; zero, so untrained it is bicubic
        self.filters = nn.Parameter(torch.zeros(3 ** 8, scale * scale * 9))

    def pattern(self, luma):
        """(N, 1, H, W) luma -> the 8 differences (N, 8, H, W), and each pixel's row (N, H, W)."""
        d = self.differences(luma)
        signs = torch.floor(d * (255 / (2 * self.dead_zone)) + 0.5).clamp(-1, 1)   # -1, 0 or 1 per neighbour; WebGPU has no Round
        low, high = self.to_halves(signs).unbind(1)
        return d, (low + 81 * high).to(torch.int32)                       # 0 .. 3^8 - 1, exact in float32; WebGPU has no integer Mul

    def residual(self, x):
        luma = self.to_luma(x).clamp(0, 1)
        d, row = self.pattern(luma)
        n, _, h, w = d.shape
        weights = self.filters.index_select(0, row.flatten()).view(n, h, w, -1, 9).permute(0, 3, 4, 1, 2)
        correction = (weights[:, :, :8] * d[:, None]).sum(dim=2) + weights[:, :, 8]   # 8 weights, then the bias
        return torch.cat([correction, correction, correction], dim=1)      # on every colour: Y moves


MODELS = {   # name -> (class, arguments): the cheapest and the best binned tables, and the 3x3 pattern filter
    "S": (FastLut, {"kernels": "S", "stages": 1}),
    "SDY x2": (FastLut, {"kernels": "SDY", "stages": 2}),
    "3x3 pattern": (PatternFilter, {"dead_zone": DEAD_ZONE}),
}


def build(name):
    """A fresh, untrained model."""
    model_class, arguments = MODELS[name]
    return model_class(scale=SCALE, **arguments).to(DEVICE)


def slug(name):
    return name.replace(" ", "")

In [ ]:
"""Checks: untrained every model is bicubic, a binned table turns with its input, and every
address is the one worked out by hand."""

models.check_skip_is_bicubic(SCALE)
lr_probe = next(iter(val_loader))[0][:8].to(DEVICE)
with torch.no_grad(), training.full_precision():
    bicubic = trim(models.bicubic(lr_probe, SCALE))
    for name in MODELS:
        model = build(name)
        assert model.halo <= manifest["halo"], f"{name} reaches further than the patches' halo"
        gap = (trim(model(lr_probe)) - bicubic).abs().max().item()
        assert gap < 1e-5, f"untrained {name} is not bicubic ({gap:.1e})"
        turned = "-"
        if isinstance(model, FastLut):   # its rotations share one table, so it turns with its input
            for table in model.tables:
                table.normal_(0, 0.05)   # random answers, so a turn shows
            grey = torch.randint(0, 256, (1, 3, 40, 40), device=DEVICE) / 255
            gap_turned = (model(torch.rot90(grey, 1, (2, 3))) - torch.rot90(model(grey), 1, (2, 3))).abs().max().item()
            assert gap_turned < 1e-5, f"{name} does not turn with its input ({gap_turned:.1e})"
            turned = f"{gap_turned:.0e}"
        entries = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"{name:<12} halo {model.halo}, untrained against bicubic {gap:.0e}, turned {turned}; "
              f"{entries:,} entries: {entries * 2 / 1e6:.2f} MB in float16")

# the rows read on a grey 8-bit picture, against ones worked out by hand
grey = torch.randint(0, 256, (1, 1, 12, 12), device=DEVICE) / 255
pixel = (grey[0, 0] * 255).round().long().cpu()
binned = build("S").address(0, grey)[0].cpu()
_, patterns = build("3x3 pattern").pattern(grey)
patterns = patterns[0].cpu()
wrong = 0
for y in range(2, 10):
    for x in range(2, 10):
        for g in range(ROTATIONS):   # the binned table: centre level and binned differences, per rotation
            values = [pixel[y + dy, x + dx].item() for dy, dx in (rotate(tap, g) for tap in KERNELS["S"])]
            expected = values[0] * CENTRE_LEVELS // 256 * BINS ** 3
            for t in (1, 2, 3):
                expected += difference_bin(torch.tensor(values[t] - values[0])).item() * BINS ** (3 - t)
            wrong += int(binned[g, y, x].item() != expected)
        expected = 0                 # the pattern: each neighbour's sign beyond the dead zone, base 3
        for i, (dy, dx) in enumerate(NEIGHBOURS):
            difference = pixel[y + dy, x + dx].item() - pixel[y, x].item()
            sign = 0 if abs(difference) < DEAD_ZONE else 1 if difference > 0 else -1
            expected += (sign + 1) * 3 ** i
        wrong += int(patterns[y, x].item() != expected)
assert wrong == 0, f"{wrong} rows differ from the hand-worked ones"
print("every row is the one worked out by hand")

## 3. Speed

Counted the way notebooks 3, 4 and 8 to 10 count: operations and memory traffic in the
exported ONNX graph at the client's tile, per LR pixel, and the roofline estimate for a
1080p frame at the share of its peak this computer's GPU reaches on notebook 7's model in
torch. The graphs are untrained: the work does not depend on the weights. Notebook 10's
interpolated tables are its section 3's numbers.

Besides `Conv`, `Add`, `Concat` and `DepthToSpace`, a binned table's graph needs `Clip`,
`Cast` to integers, integer `Add`s, and two `Gather`s - one into the index
table, one into the answers; the pattern filter needs `Floor`, `Clip`, `Cast`,
one `Gather`, and `Mul` and `ReduceSum` to apply its filter. The `Gather`s
are the idea itself; all of these are outside the budget the browser has been measured on.

In [ ]:
NOTEBOOK_10 = {   # measured in notebook 10, section 3: (ops / LR px, bytes / LR px, weights in MB)
    "notebook 10: S table": (3_850, 5_696, 1.34),
    "notebook 10: SDY x2 table": (16_724, 32_044, 5.03),
}
plan = tiles.plan_tiling((FRAME_WIDTH, FRAME_HEIGHT), scale=SCALE, halo=manifest["halo"])
tile_shape = plan["input"][2:]   # (height, width) of one tile, halo included
frame_lr = torch.rand(1, 3, FRAME_HEIGHT // SCALE, FRAME_WIDTH // SCALE, device=DEVICE)


def frame_work(cost, pixels, runs):
    """Operations and bytes for `pixels` LR pixels, computed in `runs` runs of the graph."""
    return {"operations": pixels * cost["ops"], "bytes": pixels * cost["bytes"] + runs * cost["weight_bytes"]}


def cost(model, name):
    """Operations and bytes per LR pixel of a model's graph at the client's tile, and the
    milliseconds of a 1080p frame through it in torch on this computer."""
    model = model.to(DEVICE).eval()
    graph = webexport.export(model, f"speed_check_{name}.onnx", tile_shape, halo=model.halo, models_dir=SPEED_DIR)
    work = webexport.count_work(graph["path"])
    with torch.no_grad():
        measured_ms = speed.time_ms(lambda: model(frame_lr), seconds=1.0)
    return {"ops": work["operations"] / work["input_pixels"], "bytes": work["activation_bytes"] / work["input_pixels"],
            "weight_bytes": work["weight_bytes"], "measured_ms": measured_ms, "graph": graph}


costs = {"notebook 7": cost(models.LumaUpscaler((24,), 2, 3, scale=SCALE), "luma_24x2")}
for name in MODELS:
    costs[name] = cost(build(name), f"fastlut_{slug(name)}")
for name, (ops, bytes_, megabytes) in NOTEBOOK_10.items():
    costs[name] = {"ops": ops, "bytes": bytes_, "weight_bytes": megabytes * 1e6, "measured_ms": None, "graph": None}

# calibrate on notebook 7's model, run as one whole frame
reference = costs["notebook 7"]
whole_frame = frame_work(reference, frame_lr.shape[2] * frame_lr.shape[3], runs=1)
efficiency = (speed.roofline_ms(whole_frame, *speed.HARDWARE[speed.THIS_GPU])[0] / reference["measured_ms"]
              if DEVICE.type == "cuda" else 1.0)
print(f"estimates at {efficiency:.0%} of each device's peak (notebook 7's model here)\n")
print(f"{'':<28}{'ops / LR px':>12}{'bytes / LR px':>15}{'weights, MB':>13}{'here, ms':>10}"
      + "".join(f"{short + ', ms':>14}" for short in ESTIMATE_ON))
tiled_pixels = plan["runs"] * tile_shape[0] * tile_shape[1]
for name, c in costs.items():
    estimates = [speed.roofline_ms(frame_work(c, tiled_pixels, plan["runs"]), *speed.HARDWARE[device])
                 for device in ESTIMATE_ON.values()]
    here = f"{c['measured_ms']:10.2f}" if c["measured_ms"] is not None else f"{'-':>10}"
    print(f"{name:<28}{c['ops']:12,.0f}{c['bytes']:15,.0f}{c['weight_bytes'] / 1e6:13.2f}{here}"
          + "".join(f"{ms / efficiency:7.1f} ({limit[0]})".rjust(14) for ms, limit in estimates))
print("\n(c) compute-bound, (m) memory-bound")
for c in costs.values():
    if c["graph"] is not None:
        print(webexport.summary_line(c["graph"]))

## 4. Train

The recipe of notebooks 2 to 10: Adam from 1e-3, batch 32, `metrics.ssim_l1_loss`, the epoch
with the best luma SSIM kept, each run trained until it has converged. What trains is the
table itself: every pixel reads one entry, and the loss reaches exactly the entries that
were read. `S` is trained at three seeds - their spread is what luck alone moves a score -
and the others at one.

Each run saves its training to `checkpoints/fastlut_runs/` after every epoch: run the cell
again after an interruption and it carries on; finished runs just load.

In [ ]:
def run_key(name):
    """A model's name below: S is trained at three seeds, and its seed 0 stands for it."""
    return f"S, seed {SEEDS[0]}" if name == "S" else name


def train(name, seed=0):
    """Train a fresh model, or load its saved training. Returns it at its best epoch, its
    history and its best epoch."""
    torch.manual_seed(seed)
    model = build(name)
    torch.manual_seed(seed)   # the same batches in the same order, whatever the model
    run = f"{slug(name)}_seed{seed}"
    history, best = training.train(model, train_loader, val_loader, trim, MAX_EPOCHS, learning_rate=LEARNING_RATE,
                                   loss_fn=metrics.ssim_l1_loss, select="ssim", checkpoint=RUN_DIR / f"{run}.pt",
                                   verbose=0)
    print(f"{run:<20} best epoch {best['epoch']:3} of {len(history):3}, ssim-y {best['score']:.4f}")
    return model, history, best


trained, histories, bests = {}, {}, {}
for seed in SEEDS:
    key = f"S, seed {seed}"
    trained[key], histories[key], bests[key] = train("S", seed)
for name in list(MODELS)[1:]:
    trained[name], histories[name], bests[name] = train(name)

# notebook 4's winner, the same body as notebook 7's: the model to beat
before = training.load_checkpoint(BEFORE)
trained["notebook 4's model"] = getattr(models, before["architecture"])(scale=SCALE, **before["arguments"]).to(DEVICE)
trained["notebook 4's model"].load_state_dict(before["state_dict"])
print(f"notebook 4's model: its {before['strategy']}, loaded")

### The result

Every model on the whole validation set - the 128 px cores, as every notebook scores - with
the operations its graph takes per LR pixel and its table in float16. **better** / **worse**
means a gap wider than the spread of `S`'s three seeds; **luck** means this set cannot tell
the two apart. For the same shapes, notebook 10's fine-tuned interpolated tables score
0.9251 (`S`) and 0.9343 (`SDY x2`) ssim-y on this set, and notebook 4's model 0.9292.

In [ ]:
BASELINES = {"bicubic": models.Resize(SCALE), "nearest": models.Resize(SCALE, "nearest")}


def score(name, indices=None):
    """`metrics.report` of one model (or baseline) on the validation patches `indices` (None: all)."""
    dataset = val_loader.dataset if indices is None else Subset(val_loader.dataset, indices)
    model = BASELINES[name] if name in BASELINES else trained[name]
    return training.evaluate(model.to(DEVICE), DataLoader(dataset, batch_size=128), trim)


def cost_columns(name, scores):
    """The last columns: operations per LR pixel of the model's graph, and its MB in float16."""
    if name in BASELINES:
        return ""
    row = costs["notebook 7" if name == "notebook 4's model" else name.split(",")[0]]   # a seed costs what S does
    return f"{row['ops']:12,.0f}{row['weight_bytes'] / 2e6:9.2f}"


def compare(a, b, label):
    word = "luck" if abs(a - b) <= spread else ("better" if a > b else "worse")
    print(f"{label:<40}{a - b:+.4f} ssim-y  {word}")


results = {name: score(name) for name in [*BASELINES, *trained]}
seeds = [results[f"S, seed {seed}"]["ssim"] for seed in SEEDS]
spread = max(seeds) - min(seeds)
print(f"S at {len(SEEDS)} seeds: {spread:.4f} ssim-y apart. Last columns: operations per LR pixel, and MB in float16.")
metrics.print_table(results, extra=cost_columns)

best_name = max(MODELS, key=lambda name: results[run_key(name)]["ssim"])
print()
compare(float(np.mean(seeds)), results["bicubic"]["ssim"], "S (mean of its seeds) over bicubic")
for name in list(MODELS)[1:]:
    compare(results[name]["ssim"], float(np.mean(seeds)), f"{name} over S")
compare(results["notebook 4's model"]["ssim"], results[run_key(best_name)]["ssim"], f"notebook 4's model over {best_name}")

### By content

The gain over bicubic on screens, photos and video apart: luma SSIM, and luma PSNR in dB.

In [ ]:
SHOW = [*(run_key(name) for name in MODELS), "notebook 4's model"]
print(f"{'':<8}{'patches':>8}{'bicubic ssim-y':>16}" + "".join(f"{name:>22}" for name in SHOW))
for value, indices in by_content.items():
    base = score("bicubic", indices)
    row = f"{value:<8}{len(indices):8,}{base['ssim']:16.4f}"
    for name in SHOW:
        scores = score(name, indices)
        row += f"{scores['ssim'] - base['ssim']:+.4f} / {scores['y'] - base['y']:+.2f}".rjust(22)
    print(row)

### What it looks like

Four held-out screens: bicubic, the best model here, notebook 4's model and the target. A
binned table answers in steps, so look for banding on smooth gradients as well as at the text.

In [ ]:
shown = by_content["screen"][::max(1, len(by_content["screen"]) // 4)][:4]
lr = torch.stack([val_loader.dataset[i][0] for i in shown]).to(DEVICE)
hr = torch.stack([val_loader.dataset[i][1] for i in shown])
with torch.no_grad():
    training.show_patches({"bicubic": trim(models.bicubic(lr, SCALE)),
                           best_name: trim(trained[run_key(best_name)].eval()(lr)),
                           "notebook 4's model": trim(trained["notebook 4's model"].eval()(lr)),
                           "target": hr})

## 5. Save and reload

The best model by luma SSIM is saved with everything needed to rebuild it - its tables are
its weights. We load it into a fresh model and check it computes the same thing.

In [ ]:
best_key = run_key(best_name)
best = trained[best_key]
training.save_checkpoint(CHECKPOINT, best, architecture=type(best).__name__, arguments=MODELS[best_name][1],
                         centre_levels=CENTRE_LEVELS, edges=list(EDGES), scale=SCALE, halo=best.halo,
                         loss="ssim_l1_loss", select="ssim", epochs=len(histories[best_key]),
                         best_epoch=bests[best_key]["epoch"], val=results[best_key], val_bicubic=results["bicubic"],
                         seed_spread=spread, degradation=manifest["degradation"])

# rebuild it from the file alone, the way a later notebook would
checkpoint = training.load_checkpoint(CHECKPOINT)
model_class = {"FastLut": FastLut, "PatternFilter": PatternFilter}[checkpoint["architecture"]]
loaded = model_class(scale=checkpoint["scale"], **checkpoint["arguments"]).to(DEVICE)
loaded.load_state_dict(checkpoint["state_dict"])
training.assert_same_model(best, loaded)
print("reloaded model is identical")

## 6. Export for the browser

Every model, at the client's tile (a 320x180 step) and as a whole 1080p frame: each is a
tier. A graph reshapes its look-up at the size it was exported at, so the whole-frame graph is
exported at the frame's size.

**float32 only.** An address is only exact if every tap rounds to the right 8-bit step. In
float16 a value can round to the step beside it - and a second stage reads a corrected luma,
which is no 8-bit value at all - so a float16 graph reads the neighbouring row for about 2%
of `SDY x2`'s values. That is not a model worth shipping, so every model ships in float32 -
the pattern filter's float16 graph would pass, but its table is small anyway.

**The check allows for rounding.** Even in float32, a tap that lands exactly between two
steps can round one way in torch and the other in a graph, and read the neighbouring row: the
same kind of error as the binning itself. So a graph passes if fewer than 0.5% of its output
values differ from torch's by more than the usual tolerance.

In [ ]:
def input_shape(step_height, step_width, halo):
    """The model input for a step: the step plus the halo on every side, as (height, width)."""
    return (step_height + 2 * halo, step_width + 2 * halo)


def check_and_time(info, module, shape, tolerance=2e-3, allowed=0.005):
    """Check a written graph against the torch module on an 8-bit picture - at most `allowed`
    of its values more than `tolerance` apart - then time it on the CPU and print one line."""
    picture = np.round(np.random.rand(1, 3, *shape) * 255).astype(np.float32) / 255
    with torch.no_grad():
        expected = copy.deepcopy(module).cpu()(torch.from_numpy(picture)).numpy()
    apart = np.abs(webexport.run(info["path"], picture) - expected) > tolerance
    assert apart.mean() <= allowed, f"{info['path'].name}: {apart.mean():.2%} of values apart"
    ms = webexport.time_graph(info["path"], shape)
    webexport.annotate(info, cpu_ms=round(ms, 2))
    print(webexport.summary_line(info, ms), f"  {apart.mean():.3%} of values apart")


tile_name = f"tile{STEP_WIDTH}x{STEP_HEIGHT}"
for name in MODELS:
    tag = f"fastlut_{slug(name).lower()}"
    module = trained[run_key(name)].eval()
    tile_shape = input_shape(STEP_HEIGHT, STEP_WIDTH, module.halo)
    frame_shape = input_shape(FRAME_HEIGHT // SCALE, FRAME_WIDTH // SCALE, module.halo)

    tile = webexport.export(module, f"upscale_{tag}_{tile_name}.onnx", tile_shape, label=f"{tag} fp32", halo=module.halo)
    check_and_time(tile, module, tile_shape)

    frame = webexport.export(module, f"upscale_{tag}_{tile_name}_frame.onnx", frame_shape,
                             label=f"{tag} fp32 · whole frame", halo=module.halo)
    check_and_time(frame, module, frame_shape)
print("compare in a browser: uv run src/upscale/benchmark/main.py")